## Versión final organizada

Esta copia escribe sus salidas bajo `outputs/`. Las rutas comunes se definen en `project_paths.py`.


In [ ]:
from project_paths import PROJECT_ROOT, ANALYSIS_DATA_DIR, OSSE_DATA_DIR, OSSE_RESULTS_DIR, OSSE_FIGURES_DIR, ensure_output_directories
ensure_output_directories()


## Run Observing System Simulation Experiment (OSSE)

Notebook derived from `run_cse.ipynb`, but without control interventions. It first selects the full truth/observation window, then runs sequential DA cycles and ensemble forecasts.


In [ ]:
#-----------------------------------------------------------
# Import libraries and project classes:
#-----------------------------------------------------------
# This notebook uses NumPy for array work, pickle for persisted OSSE/CSE
# objects, pathlib for file paths, and Matplotlib for diagnostic figures.
import numpy as np
import pickle
import sys
from pathlib import Path
import matplotlib.pyplot as plt

# Project-specific Lorenz-63, state-vector, observation, DA, and OSSE classes.
from class_Lorenz63 import run_RK4
from class_state_vec import state_vector
from class_obs import obs_da
from class_da_sys import da_system

# Reload class_osse so notebook reruns pick up recent edits without restarting
# the Python kernel.
import importlib
import class_osse
importlib.reload(class_osse)
from class_osse import osse


In [ ]:
#-----------------------------------------------------------
# Define model, load DA configuration, truth, and observations:
#-----------------------------------------------------------
# Lorenz-63 physical parameters. The same model function is used for the
# Nature Run, OSSE forecasts, and plotting helpers.
sigma = 10.0       # Lorenz-63 sigma parameter
rho   = 28.0       # Lorenz-63 rho parameter
beta  = 8.0 / 3.0  # Lorenz-63 beta parameter

lorenz63_model = osse.make_lorenz63_model(sigma=sigma, rho=rho, beta=beta)

# Load the preconfigured DA system. This object contains the initial ensemble
# settings, observation operator/covariance, state vector, observations, and
# time-step metadata used throughout the OSSE.
name   = 'x_analysis'
infile = str(ANALYSIS_DATA_DIR / (name + '_init.pkl'))
das    = da_system()
das    = das.load(infile)
#das.ens_sigma_init = np.sqrt(2)
#print(das)

# Build and print the initial ensemble from the DA-system settings. This is a
# quick sanity check; the OSSE run itself rebuilds its own initial ensemble from
# x_nature[:, t0] inside class_osse.
xa = das.x0
Xa = das.initEns(xa, mu = das.ens_bias_init, sigma = das.ens_sigma_init, edim = das.edim, separate = 'no')

print('Ensemble dimension: ', das.edim)
print('Initial bias: ', das.ens_bias_init)
print('Initial standard deviation: ', das.ens_sigma_init)
print('xa: ', xa)
print('Xa: ', Xa)

# Load the full Nature Run trajectory. This is the truth trajectory used by the
# OSSE, with shape (state_dimension, time_index).
sv       = das.getStateVector()
x_nature = sv.getTrajectory()

#print('x_nature: ', x_nature)

# Load the precomputed synthetic observations corresponding to the Nature Run.
# y_obs is indexed on the same absolute time axis as x_nature.
obs   = das.getObsData()
y_obs = obs.getVal()
y_pos = obs.getPos()
y_err = obs.getErr()

# Cache frequently used DA/time metadata from the loaded DA system. These names
# are used by later notebook cells and by the OSSE wrapper.
t_nature  = sv.getTimes()
acyc_step = das.acyc_step
dtau      = das.dtau
fcst_step = das.fcst_dt
fcst_dt   = das.fcst_dt
maxit     = das.maxit
xdim      = das.xdim
ydim      = das.ydim

print('das.dt: ', das.dt)
print('das.fcst_dt: ', das.fcst_dt)

# Confirm the DA method, typically EnKF/ETKF in this workflow.
method = das.getMethod()
print('Method: ', method)


#### PREPARE OSSE START TIME:


In [ ]:
#-----------------------------------------------------------
# PREPARE OSSE START TIME:
#-----------------------------------------------------------
# The current OSSE workflow always starts at the beginning of the Nature Run.
# There is no CSE event-index selection here: t0=0 is the absolute time origin
# for the OSSE, and the full-period run covers the available Nature Run from
# that point onward.
multiple_number = 4
position = int(np.ceil(75.1 * multiple_number))

OSSE_START_AT_NATURE_T0 = True
t_start = 0
t0 = 0

# Store the truth state and observation at the OSSE start.
x_original = x_nature[:, t0].copy()
obs_original = y_obs[:, t0].copy()

print('OSSE starts literally at Nature Run t=0.')
print(f'OSSE t_start = {t_start}')
print(f'OSSE aligned t0 = {t0}')
print(f'Initial condition: {x_original}')


#### Plot nature run and observations around the OSSE start:


In [ ]:
#-----------------------------------------------------------
# Plot nature run and observations around the OSSE start:
#-----------------------------------------------------------
# This quick-look figure checks that the truth trajectory and synthetic
# observations are aligned around the OSSE start time.
PathFigures = str(OSSE_FIGURES_DIR) + '/'
Path(PathFigures).mkdir(parents=True, exist_ok=True)
sv.setFigurePath(PathFigures)

# Plot the first 350 model time steps after t0. The observation frequency is
# expressed in model time-step indices; here obs_freq=8 matches Ta below.
window   = 350
tf       = t0 + window
obs_freq = 8
print('t0, tf: ', t0, tf)

# Reuse the Lorenz-63 plotting helper to draw truth and observations only.
# Later cells add the OSSE analysis/prior ensemble members to a similar view.
plot_solver = run_RK4(lorenz63_model, sigma, beta, rho, x_nature[:, t0], fcst_dt, fcst_dt)
plot_solver.plot_nature_obs(x_nature, y_obs, t0, tf, obs_freq, PathFigures)


####  Run standard OSSE experiment:


In [ ]:
#-----------------------------------------------------------
# Run standard OSSE experiment:
#-----------------------------------------------------------
"""
The Observing System Simulation Experiment (OSSE) follows these steps:

1. Start from the Nature Run at absolute time t0=0.
2. Use the precomputed synthetic observations at DA times.
3. At each DA cycle, assimilate observations to produce an analysis ensemble Xa.
4. Forecast the ensemble forward to the next DA time to produce the next prior Xf.
5. Save the histories needed by later CSE fallback experiments.

With RUN_OSSE_MINIMAL_OUTPUT=True, the long full-period run stores only the
analysis/prior histories and summaries, avoiding large diagnostic objects.
"""

# Parameters to edit:
Ta = 8                         # Number of model time steps between DA cycles.
DEFAULT_OSSE_NUM_CYCLES = 1000 # Used only when RUN_OSSE_FULL_NATURE_PERIOD=False.
RUN_OSSE_FULL_NATURE_PERIOD = True
RUN_OSSE_MINIMAL_OUTPUT = True
OSSE_PROGRESS_EVERY = 10000    # Print progress every N completed DA cycles.
OSSE_CHECKPOINT_EVERY = 50000  # Flush checkpoint files every N completed cycles.
multiple_number = 4            # Forecast horizon multiplier used by the full OSSE path.
T_0 = 75.1                     # Base forecast horizon used by the original workflow.
#MULTIPLICATIVE_INFLATION = 1.015
MULTIPLICATIVE_INFLATION = 1.04
#MULTIPLICATIVE_INFLATION = 1.05

# Output naming convention: Xa = analysis ensemble; Xf = prior/forecast ensemble
# at each DA time. The .npy files are incremental checkpoints; the .pkl files are
# final persisted histories used by downstream notebooks.
OSSE_OUTPUT_FILES = {
    'analysis_file': str(OSSE_DATA_DIR / 'osse_Xa_history.pkl'),
    'prior_file': str(OSSE_DATA_DIR / 'osse_Xf_history.pkl'),
    'summary_file': str(OSSE_DATA_DIR / 'osse_summary.pkl'),
    'checkpoint_file': str(OSSE_DATA_DIR / 'osse_Xa_history_checkpoint.npy'),
    'prior_checkpoint_file': str(OSSE_DATA_DIR / 'osse_Xf_history_checkpoint.npy'),
    'checkpoint_summary_file': str(OSSE_DATA_DIR / 'osse_summary_checkpoint.pkl'),
}

# Construct the OSSE experiment object. It holds the DA system, truth,
# observations, model function, model parameters, and plotting path.
osse_exp = osse(
    das=das,
    x_nature=x_nature,
    y_obs=y_obs,
    model_func=lorenz63_model,
    sigma=sigma,
    beta=beta,
    rho=rho,
    fcst_dt=fcst_dt,
    path_figures=PathFigures,
)

# Run the OSSE. The class method decides the number of cycles, performs the DA
# loop, writes checkpoints/final files in minimal mode, and returns a dictionary
# with the variables needed by later cells.
osse_run = osse_exp.run_standard_osse_experiment(
    t0=t0,
    Ta=Ta,
    default_num_cycles=DEFAULT_OSSE_NUM_CYCLES,
    run_full_nature_period=RUN_OSSE_FULL_NATURE_PERIOD,
    run_minimal_output=RUN_OSSE_MINIMAL_OUTPUT,
    progress_every=OSSE_PROGRESS_EVERY,
    checkpoint_every=OSSE_CHECKPOINT_EVERY,
    multiple_number=multiple_number,
    T_0=T_0,
    multiplicative_inflation=MULTIPLICATIVE_INFLATION,
    **OSSE_OUTPUT_FILES,
    extra_summary={
        'input_das_file': infile,
        'osse_start_at_nature_t0': bool(globals().get('OSSE_START_AT_NATURE_T0', False)),
        'run_cse_event_index': None,
        't_start': t_start,
        'tf': tf,
        'obs_freq': obs_freq,
        'path_figures': PathFigures,
        'das_edim': int(das.edim),
        'das_ens_sigma_init': float(das.ens_sigma_init),
        'das_ens_bias_init': float(das.ens_bias_init),
        'das_R_variance': float(das.R[0, 0]),
        'initEns_seed': 2,
    },
)

# Publish only selected outputs into the notebook namespace. This keeps later
# cells convenient without doing an unrestricted globals().update(osse_run).
osse.publish_run_outputs(
    osse_run,
    globals(),
    include_full_diagnostics=not RUN_OSSE_MINIMAL_OUTPUT,
)


#### PLOT OSSE NATURE, OBSERVATIONS, AND ANALYSIS MEMBERS


In [ ]:
#--------------------------------------------------------------------
# PLOT OSSE NATURE, OBSERVATIONS, AND ANALYSIS/PRIOR MEMBERS
#--------------------------------------------------------------------
# This diagnostic plot overlays the Nature Run, synthetic observations,
# analysis ensemble members (Xa), and prior ensemble members (Xf) over a short
# time window. It reads the saved minimal-output histories from disk.
fig, axes, outfile = osse_exp.plot_nature_observations_analysis_members(
    summary_file=str(OSSE_DATA_DIR / 'osse_summary.pkl'),
    plot_start=0,
    plot_window=350,
    output_dir=PathFigures,
)


## MULTIPLICATIVE INFLATION SWEEP TABLE


In [ ]:
#--------------------------------------------------------------------
# MULTIPLICATIVE INFLATION SWEEP TABLE
#--------------------------------------------------------------------
# This optional diagnostic reruns short OSSE experiments for several inflation
# values and compares total RMSE and ensemble spread. It is intentionally
# separate from the long full-period minimal OSSE.

# Parameters to edit:
RUN_INFLATION_SWEEP = True
INFLATION_SWEEP_NUM_CYCLES = 1000
INFLATION_SWEEP_VALUES = np.round(np.arange(1.01, 1.09 + 1e-12, 0.005), 3)
INFLATION_SWEEP_SEED = 2
INFLATION_SWEEP_CSV_FILE = str(OSSE_RESULTS_DIR / 'osse_inflation_sweep_table.csv')
INFLATION_SWEEP_TABLE_FIGURE_FILE = PathFigures + 'osse_inflation_sweep_table.png'
INFLATION_SWEEP_CURVE_FIGURE_FILE = PathFigures + 'osse_inflation_sweep_rmse_spread.png'

if RUN_INFLATION_SWEEP:
    inflation_table, (fig_table, ax_table), (fig, ax) = osse_exp.run_inflation_sweep_table(
        t0=t0,
        Ta=Ta,
        num_cycles=INFLATION_SWEEP_NUM_CYCLES,
        multiple_number=multiple_number,
        T_0=T_0,
        inflation_values=INFLATION_SWEEP_VALUES,
        seed=INFLATION_SWEEP_SEED,
        csv_file=INFLATION_SWEEP_CSV_FILE,
        table_figure_file=INFLATION_SWEEP_TABLE_FIGURE_FILE,
        curve_figure_file=INFLATION_SWEEP_CURVE_FIGURE_FILE,
        display_table=True,
    )
    plt.show()
else:
    print('RUN_INFLATION_SWEEP is False. Skipping inflation sweep.')


## Stable Multiplicative Inflation Sweep Table


In [ ]:
#--------------------------------------------------------------------
# STABLE MULTIPLICATIVE INFLATION SWEEP TABLE
#--------------------------------------------------------------------
# This cell is self-contained enough to run without rerunning the full notebook.
# If the OSSE objects are already in memory, it reuses them. Otherwise, it loads
# the DA system, Nature Run, and observations from x_analysis_init.pkl and
# reconstructs osse_exp.

# Parameters to edit:
RUN_STABLE_INFLATION_SWEEP = True
INFLATION_SWEEP_NUM_CYCLES = 8000
INFLATION_SWEEP_VALUES = np.arange(1.01, 1.06 + 1e-12, 0.01)
INFLATION_SWEEP_SPINUP_EXCLUDE = 100
INFLATION_SWEEP_SEED = 2
INFLATION_SWEEP_CSV_FILE = str(OSSE_RESULTS_DIR / 'osse_inflation_sweep_table_stable.csv')

# Self-contained setup:
from pathlib import Path
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt

BASE_PATH = ANALYSIS_DATA_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from class_da_sys import da_system
import class_osse
class_osse = importlib.reload(class_osse)
from class_osse import osse

sigma = globals().get('sigma', 10.0)
rho = globals().get('rho', 28.0)
beta = globals().get('beta', 8.0 / 3.0)
lorenz63_model = osse.make_lorenz63_model(sigma=sigma, rho=rho, beta=beta)

if not all(name in globals() for name in ['das', 'x_nature', 'y_obs']):
    infile = str(BASE_PATH / 'x_analysis_init.pkl')
    das = da_system().load(infile)
    sv = das.getStateVector()
    x_nature = sv.getTrajectory()
    obs = das.getObsData()
    y_obs = obs.getVal()
else:
    infile = globals().get('infile', str(BASE_PATH / 'x_analysis_init.pkl'))

fcst_dt = globals().get('fcst_dt', das.fcst_dt)
Ta = globals().get('Ta', 8)
t0 = globals().get('t0', 0)
multiple_number = globals().get('multiple_number', 4)
T_0 = globals().get('T_0', 75.1)
PathFigures = globals().get('PathFigures', str(OSSE_FIGURES_DIR) + '/')
Path(PathFigures).mkdir(parents=True, exist_ok=True)

INFLATION_SWEEP_TABLE_FIGURE_FILE = PathFigures + 'osse_inflation_sweep_table_stable.png'
INFLATION_SWEEP_CURVE_FIGURE_FILE = PathFigures + 'osse_inflation_sweep_rmse_spread_stable.png'

osse_exp = osse(
    das=das,
    x_nature=x_nature,
    y_obs=y_obs,
    model_func=lorenz63_model,
    sigma=sigma,
    beta=beta,
    rho=rho,
    fcst_dt=fcst_dt,
    path_figures=PathFigures,
)

if RUN_STABLE_INFLATION_SWEEP:
    inflation_table_stable, (fig_table_stable, ax_table_stable), (fig_stable, ax_stable) = osse_exp.run_inflation_sweep_table(
        t0=t0,
        Ta=Ta,
        num_cycles=INFLATION_SWEEP_NUM_CYCLES,
        multiple_number=multiple_number,
        T_0=T_0,
        inflation_values=INFLATION_SWEEP_VALUES,
        seed=INFLATION_SWEEP_SEED,
        csv_file=INFLATION_SWEEP_CSV_FILE,
        table_figure_file=INFLATION_SWEEP_TABLE_FIGURE_FILE,
        curve_figure_file=INFLATION_SWEEP_CURVE_FIGURE_FILE,
        display_table=True,
        spinup_exclude=INFLATION_SWEEP_SPINUP_EXCLUDE,
    )
    plt.show()
else:
    print('RUN_STABLE_INFLATION_SWEEP is False. Skipping stable inflation sweep.')


## Save OSSE Outputs


In [ ]:
#-----------------------------------------------------------
# Save OSSE outputs:
#-----------------------------------------------------------
# In minimal mode, the OSSE histories and summary are saved inside the run loop
# itself. This cell only performs an additional full-output save when the OSSE
# was run in diagnostic/full-output mode.
if globals().get('RUN_OSSE_MINIMAL_OUTPUT', False):
    print('Minimal OSSE already saved osse_Xa_history.pkl, osse_Xf_history.pkl, and osse_summary.pkl during the run.')
else:
    summary, results = osse_exp.save_standard_osse_outputs(
        osse_results,
        diagnostics=diagnostics,
        truth_file=str(OSSE_DATA_DIR / 'osse_truth_full.pkl'),
        truth_da_file=str(OSSE_DATA_DIR / 'osse_truth_da_times.pkl'),
        obs_file=str(OSSE_DATA_DIR / 'osse_yobs.pkl'),
        summary_file=str(OSSE_DATA_DIR / 'osse_summary.pkl'),
        results_file=str(OSSE_DATA_DIR / 'osse_results.pkl'),
        analysis_file=str(OSSE_DATA_DIR / 'osse_analysis_history.pkl'),
        xa_history_file=str(OSSE_DATA_DIR / 'osse_Xa_history.pkl'),
        prior_file=str(OSSE_DATA_DIR / 'osse_Xf_history.pkl'),
        extra_summary={
            'input_das_file': infile,
            'osse_start_at_nature_t0': bool(globals().get('OSSE_START_AT_NATURE_T0', False)),
            'run_cse_event_index': None,
            't_start': t_start,
            'tf': tf,
            'obs_freq': obs_freq,
            'path_figures': PathFigures,
            'run_osse_full_nature_period': bool(globals().get('RUN_OSSE_FULL_NATURE_PERIOD', False)),
            'nature_run_last_index': int(globals().get('nature_run_last_index', -1)),
            'full_nature_period_cycles_from_osse_t0': int(globals().get('full_nature_period_cycles_from_osse_t0', -1)),
        },
    )
